# Week 2 — Data Cleaning & Quality Improvement

## Objective

The purpose of this notebook is to clean and validate the CodeTX supply-chain datasets while preserving valid business records and documenting every cleaning decision.

## Week 2 Goals

- Investigate duplicate transaction identifiers
- Preserve valid records while creating reliable unique identifiers
- Standardize data types and formats
- Validate important business rules
- Identify inconsistent or impossible values
- Create analysis-ready cleaned datasets
- Document all cleaning decisions
- Export cleaned datasets for later analysis

In [1]:
import pandas as pd
import numpy as np

In [2]:
branches = pd.read_csv("branches.csv")
customers = pd.read_csv("customers.csv")
inventory = pd.read_csv("inventory_master.csv")
invoices = pd.read_csv("invoices.csv")
payments = pd.read_csv("payments.csv")
products = pd.read_csv("products.csv")
po_header = pd.read_csv("purchase_orders_header.csv")
po_lines = pd.read_csv("purchase_orders_lines.csv")
so_header = pd.read_csv("sales_orders_header.csv")
so_lines = pd.read_csv("sales_orders_lines.csv")
stock_ledger = pd.read_csv("stock_ledger.csv")
suppliers = pd.read_csv("suppliers.csv")

In [3]:
datasets = {
    "branches": branches,
    "customers": customers,
    "inventory": inventory,
    "invoices": invoices,
    "payments": payments,
    "products": products,
    "po_header": po_header,
    "po_lines": po_lines,
    "so_header": so_header,
    "so_lines": so_lines,
    "stock_ledger": stock_ledger,
    "suppliers": suppliers
}

In [5]:
invoices["invoice_record_id"] = (
    invoices["invoice_id"].astype(str)
    + "_"
    + invoices["so_id"].astype(str)
)

payments["payment_record_id"] = (
    payments["payment_id"].astype(str)
    + "_"
    + payments["invoice_id"].astype(str)
)

In [6]:
print(
    "Invoice record duplicates:",
    invoices["invoice_record_id"].duplicated().sum()
)

print(
    "Payment record duplicates:",
    payments["payment_record_id"].duplicated().sum()
)

Invoice record duplicates: 0
Payment record duplicates: 0


### Duplicate Identifier Cleaning Decision

Week 1 identified repeated `invoice_id` and `payment_id` values that belonged to different transactions.

Because these records were not exact duplicates, no rows were deleted.

Instead, new analytical identifiers were created:

- `invoice_record_id` = `invoice_id` + `so_id`
- `payment_record_id` = `payment_id` + `invoice_id`

Both new identifiers were validated and contain zero duplicates.

This approach preserves the original transaction data while providing unique keys for reliable analysis and future joins.

In [7]:
date_columns = {
    "customers": ["customer_since", "last_purchase_date"],
    "invoices": ["invoice_date", "due_date"],
    "payments": ["payment_date"],
    "products": ["last_purchase_date"],
    "po_header": ["order_date", "expected_delivery_date", "received_date"],
    "so_header": ["order_date", "delivery_date"],
    "stock_ledger": ["movement_date"]
}

for dataset_name, columns in date_columns.items():
    for column in columns:
        datasets[dataset_name][column] = pd.to_datetime(
            datasets[dataset_name][column],
            errors="coerce"
        )

In [8]:
for dataset_name, columns in date_columns.items():
    for column in columns:
        print(
            f"{dataset_name}.{column}: "
            f"{datasets[dataset_name][column].dtype}"
        )

customers.customer_since: datetime64[us]
customers.last_purchase_date: datetime64[us]
invoices.invoice_date: datetime64[us]
invoices.due_date: datetime64[us]
payments.payment_date: datetime64[us]
products.last_purchase_date: datetime64[us]
po_header.order_date: datetime64[us]
po_header.expected_delivery_date: datetime64[us]
po_header.received_date: datetime64[us]
so_header.order_date: datetime64[us]
so_header.delivery_date: datetime64[us]
stock_ledger.movement_date: datetime64[us]


In [9]:
branches["warehouse_capacity_sqft"] = (
    branches["warehouse_capacity"]
    .str.replace(" sqft", "", regex=False)
    .astype(int)
)

In [10]:
branches[
    ["warehouse_capacity", "warehouse_capacity_sqft"]
]

,warehouse_capacity,warehouse_capacity_sqft
0,45230 sqft,45230
1,38750 sqft,38750
2,41890 sqft,41890
3,36210 sqft,36210
4,29880 sqft,29880
5,34120 sqft,34120


In [11]:
product_rules = pd.DataFrame({
    "selling_price_below_cost":
        products["unit_price"] < products["unit_cost"],

    "safety_stock_above_reorder":
        products["safety_stock"] > products["reorder_level"],

    "reorder_above_max_stock":
        products["reorder_level"] > products["max_stock_level"],

    "safety_above_max_stock":
        products["safety_stock"] > products["max_stock_level"]
})

product_rules.sum()

selling_price_below_cost      0
safety_stock_above_reorder    0
reorder_above_max_stock       0
safety_above_max_stock        0
dtype: int64

### Product Business-Rule Validation

The product master was tested against several inventory and pricing rules.

No violations were found:

- No product has a selling price below its recorded unit cost.
- No product has safety stock above the reorder level.
- No product has a reorder level above the maximum stock level.
- No product has safety stock above the maximum stock level.

These results indicate that the core product inventory thresholds and pricing fields are internally consistent.

In [12]:
inventory_rules = pd.DataFrame({
    "safety_above_reorder":
        inventory["safety_stock"] > inventory["reorder_level"],

    "reorder_above_max":
        inventory["reorder_level"] > inventory["max_stock"],

    "safety_above_max":
        inventory["safety_stock"] > inventory["max_stock"],

    "negative_current_stock":
        inventory["current_stock"] < 0,

    "current_stock_above_max":
        inventory["current_stock"] > inventory["max_stock"]
})

inventory_rules.sum()

safety_above_reorder         0
reorder_above_max            0
safety_above_max             0
negative_current_stock       0
current_stock_above_max    180
dtype: int64

### Inventory Business-Rule Validation

The inventory table passed the basic stock-threshold checks:

- No safety-stock value exceeds its reorder level.
- No reorder level exceeds the configured maximum stock.
- No safety-stock value exceeds maximum stock.
- No current-stock value is negative.

However, all 180 product-branch combinations have `current_stock` above their configured `max_stock` level.

Because current stock was previously validated against the latest stock-ledger running balance, these values are internally consistent and should not be treated as simple data-entry errors.

This finding suggests a potential organization-wide overstocking issue and should be investigated further during the inventory analytics phase rather than corrected during data cleaning.

In [13]:
po_rules = pd.DataFrame({
    "expected_before_order":
        po_header["expected_delivery_date"] < po_header["order_date"],

    "received_before_order":
        po_header["received_date"] < po_header["order_date"],

    "received_before_expected":
        po_header["received_date"] < po_header["expected_delivery_date"],

    "negative_total_cost":
        po_header["total_cost"] < 0,

    "negative_grand_total":
        po_header["grand_total"] < 0,

    "grand_total_below_cost":
        po_header["grand_total"] < po_header["total_cost"]
})

po_rules.sum()

expected_before_order         0
received_before_order         0
received_before_expected    971
negative_total_cost           0
negative_grand_total          0
grand_total_below_cost        0
dtype: int64

### Purchase Order Business-Rule Validation

Purchase-order dates and financial values were validated against several logical business rules.

No invalid cases were found for:

- expected delivery occurring before the order date
- received date occurring before the order date
- negative purchase costs
- negative grand totals
- grand totals below total purchase cost

A total of 971 purchase orders were received before their expected delivery date.

This is not considered a data-quality problem because early delivery is a valid business outcome and may indicate strong supplier performance.

These records will be preserved for later supplier reliability and delivery-performance analysis.

In [14]:
so_rules = pd.DataFrame({
    "delivery_before_order":
        so_header["delivery_date"] < so_header["order_date"],

    "negative_order_value":
        so_header["total_order_value"] < 0,

    "negative_grand_total":
        so_header["grand_total"] < 0,

    "grand_total_below_order_value":
        so_header["grand_total"] < so_header["total_order_value"]
})

so_rules.sum()

delivery_before_order            0
negative_order_value             0
negative_grand_total             0
grand_total_below_order_value    0
dtype: int64

### Sales Order Business-Rule Validation

The sales-order data was checked for invalid dates and financial inconsistencies.

No violations were found:

- No delivery date occurs before the corresponding order date.
- No negative order values were identified.
- No negative grand totals were identified.
- No grand total is below the corresponding total order value.

The sales-order data therefore appears internally consistent for these core business rules.

In [15]:
invoice_rules = pd.DataFrame({
    "due_before_invoice":
        invoices["due_date"] < invoices["invoice_date"],

    "negative_order_value":
        invoices["total_order_value"] < 0,

    "negative_grand_total":
        invoices["grand_total"] < 0,

    "grand_total_below_order_value":
        invoices["grand_total"] < invoices["total_order_value"]
})

invoice_rules.sum()

due_before_invoice               0
negative_order_value             0
negative_grand_total             0
grand_total_below_order_value    0
dtype: int64

In [16]:
payment_rules = pd.DataFrame({
    "negative_payment":
        payments["payment_amount"] < 0,

    "zero_payment":
        payments["payment_amount"] == 0
})

payment_rules.sum()

negative_payment    0
zero_payment        0
dtype: int64

In [17]:
invoice_rules.sum()

due_before_invoice               0
negative_order_value             0
negative_grand_total             0
grand_total_below_order_value    0
dtype: int64

### Invoice and Payment Business-Rule Validation

Invoice and payment records were checked for basic date and financial consistency.

No invoice violations were found:

- No invoice due date occurs before the invoice date.
- No negative invoice order values were identified.
- No negative invoice grand totals were identified.
- No invoice grand total is below its corresponding order value.

Payment records were also validated:

- No negative payment amounts were found.
- No zero-value payments were found.

These results indicate that the invoice and payment amounts are internally consistent, although the duplicate identifier issue identified earlier still requires careful handling during joins and downstream analysis.

In [18]:
po_lines_check = pd.DataFrame({
    "line_total_mismatch":
        ~np.isclose(
            po_lines["line_total"],
            po_lines["quantity"] * po_lines["unit_cost"],
            rtol=1e-5,
            atol=1e-2
        ),

    "grand_total_mismatch":
        ~np.isclose(
            po_lines["line_grand_total"],
            po_lines["line_total"] + po_lines["gst_amount"],
            rtol=1e-5,
            atol=1e-2
        )
})

po_lines_check.sum()

line_total_mismatch     0
grand_total_mismatch    0
dtype: int64

In [19]:
so_lines_check = pd.DataFrame({
    "line_total_mismatch":
        ~np.isclose(
            so_lines["line_total"],
            so_lines["quantity"] * so_lines["unit_price"],
            rtol=1e-5,
            atol=1e-2
        ),

    "grand_total_mismatch":
        ~np.isclose(
            so_lines["line_grand_total"],
            so_lines["line_total"] + so_lines["gst_amount"],
            rtol=1e-5,
            atol=1e-2
        )
})

so_lines_check.sum()

line_total_mismatch     0
grand_total_mismatch    0
dtype: int64

### Line-Level Financial Validation

Purchase-order and sales-order line calculations were validated using tolerance-based numerical comparisons.

No mismatches were found in either dataset.

For purchase-order lines:

- `line_total` correctly matches `quantity × unit_cost`.
- `line_grand_total` correctly matches `line_total + gst_amount`.

For sales-order lines:

- `line_total` correctly matches `quantity × unit_price`.
- `line_grand_total` correctly matches `line_total + gst_amount`.

These results confirm that the line-level financial calculations are internally consistent and suitable for downstream cost, revenue, and margin analysis.

In [20]:
po_line_summary = (
    po_lines
    .groupby("po_id")
    .agg(
        calculated_total_cost=("line_total", "sum"),
        calculated_gst=("gst_amount", "sum"),
        calculated_grand_total=("line_grand_total", "sum")
    )
    .reset_index()
)

po_header_validation = po_header.merge(
    po_line_summary,
    on="po_id",
    how="left"
)

po_header_validation["cost_match"] = np.isclose(
    po_header_validation["total_cost"],
    po_header_validation["calculated_total_cost"],
    rtol=1e-5,
    atol=1e-2
)

po_header_validation["gst_match"] = np.isclose(
    po_header_validation["total_gst_amount"],
    po_header_validation["calculated_gst"],
    rtol=1e-5,
    atol=1e-2
)

po_header_validation["grand_total_match"] = np.isclose(
    po_header_validation["grand_total"],
    po_header_validation["calculated_grand_total"],
    rtol=1e-5,
    atol=1e-2
)

print("PO total mismatches:",
      (~po_header_validation["cost_match"]).sum())

print("PO GST mismatches:",
      (~po_header_validation["gst_match"]).sum())

print("PO grand total mismatches:",
      (~po_header_validation["grand_total_match"]).sum())

PO total mismatches: 0
PO GST mismatches: 0
PO grand total mismatches: 0


### Purchase Order Header-to-Line Validation

Purchase-order header totals were compared with the aggregated values from the corresponding purchase-order line items.

No mismatches were found:

- Purchase-order total cost matches the sum of line totals.
- Purchase-order GST matches the sum of line GST amounts.
- Purchase-order grand total matches the sum of line grand totals.

This confirms that the purchase-order header and line datasets are financially consistent and can be safely used together in later procurement analysis.

In [22]:
so_line_summary = (
    so_lines
    .groupby("so_id")
    .agg(
        calculated_order_value=("line_total", "sum"),
        calculated_gst=("gst_amount", "sum"),
        calculated_grand_total=("line_grand_total", "sum")
    )
    .reset_index()
)

so_header_validation = so_header.merge(
    so_line_summary,
    on="so_id",
    how="left"
)

so_header_validation["order_value_match"] = np.isclose(
    so_header_validation["total_order_value"],
    so_header_validation["calculated_order_value"],
    rtol=1e-5,
    atol=1e-2
)

so_header_validation["gst_match"] = np.isclose(
    so_header_validation["total_gst_amount"],
    so_header_validation["calculated_gst"],
    rtol=1e-5,
    atol=1e-2
)

so_header_validation["grand_total_match"] = np.isclose(
    so_header_validation["grand_total"],
    so_header_validation["calculated_grand_total"],
    rtol=1e-5,
    atol=1e-2
)

print("SO order value mismatches:",
      (~so_header_validation["order_value_match"]).sum())

print("SO GST mismatches:",
      (~so_header_validation["gst_match"]).sum())

print("SO grand total mismatches:",
      (~so_header_validation["grand_total_match"]).sum())

SO order value mismatches: 0
SO GST mismatches: 0
SO grand total mismatches: 0


### Sales Order Header-to-Line Validation

Sales-order header totals were compared with aggregated values from the corresponding sales-order line items.

No mismatches were found:

- Sales-order total order value matches the sum of line totals.
- Sales-order GST matches the sum of line GST amounts.
- Sales-order grand total matches the sum of line grand totals.

This confirms that the sales-order header and line datasets are financially consistent and suitable for downstream sales and revenue analysis.

In [23]:
invoice_so_validation = invoices.merge(
    so_header[
        ["so_id", "total_order_value", "total_gst_amount", "grand_total"]
    ],
    on="so_id",
    how="left",
    suffixes=("_invoice", "_sales_order")
)

invoice_so_validation["order_value_match"] = np.isclose(
    invoice_so_validation["total_order_value_invoice"],
    invoice_so_validation["total_order_value_sales_order"],
    rtol=1e-5,
    atol=1e-2
)

invoice_so_validation["gst_match"] = np.isclose(
    invoice_so_validation["total_gst_amount_invoice"],
    invoice_so_validation["total_gst_amount_sales_order"],
    rtol=1e-5,
    atol=1e-2
)

invoice_so_validation["grand_total_match"] = np.isclose(
    invoice_so_validation["grand_total_invoice"],
    invoice_so_validation["grand_total_sales_order"],
    rtol=1e-5,
    atol=1e-2
)

print("Invoice vs SO order value mismatches:",
      (~invoice_so_validation["order_value_match"]).sum())

print("Invoice vs SO GST mismatches:",
      (~invoice_so_validation["gst_match"]).sum())

print("Invoice vs SO grand total mismatches:",
      (~invoice_so_validation["grand_total_match"]).sum())

Invoice vs SO order value mismatches: 0
Invoice vs SO GST mismatches: 0
Invoice vs SO grand total mismatches: 0


### Invoice-to-Sales-Order Validation

Invoice totals were compared with the corresponding sales-order totals using `so_id`.

No mismatches were found:

- Invoice order value matches the related sales-order value.
- Invoice GST matches the related sales-order GST.
- Invoice grand total matches the related sales-order grand total.

This confirms that invoices are financially consistent with their linked sales orders.

In [24]:
payment_summary = (
    payments
    .groupby("invoice_id")
    .agg(
        total_paid=("payment_amount", "sum"),
        payment_count=("payment_id", "count")
    )
    .reset_index()
)

invoice_payment_validation = invoices.merge(
    payment_summary,
    on="invoice_id",
    how="left"
)

invoice_payment_validation["total_paid"] = (
    invoice_payment_validation["total_paid"]
    .fillna(0)
)

invoice_payment_validation["payment_gap"] = (
    invoice_payment_validation["grand_total"]
    - invoice_payment_validation["total_paid"]
)

invoice_payment_validation[
    ["payment_status", "grand_total", "total_paid", "payment_gap"]
].head()

,payment_status,grand_total,total_paid,payment_gap
0,Unpaid,1720159.6,0.0,1720159.6
1,Unpaid,2804273.0,0.0,2804273.0
2,Paid,3354624.0,3354624.0,0.0
3,Paid,1387911.0,1387911.0,0.0
4,Paid,1475.0,1475.0,0.0


In [25]:
payment_status_summary = (
    invoice_payment_validation
    .groupby("payment_status")
    .agg(
        invoice_count=("invoice_id", "count"),
        avg_invoice_value=("grand_total", "mean"),
        avg_total_paid=("total_paid", "mean"),
        avg_payment_gap=("payment_gap", "mean")
    )
    .round(2)
)

payment_status_summary

,invoice_count,avg_invoice_value,avg_total_paid,avg_payment_gap
payment_status,,,,
Paid,12536,1619388.30,1648903.57,-29515.27
Partially Paid,3660,1630904.36,849991.94,780912.42
Unpaid,1837,1624513.59,24715.69,1599797.90


In [26]:
invoice_id_counts = (
    invoices["invoice_id"]
    .value_counts()
)

unique_invoice_ids = invoice_id_counts[
    invoice_id_counts == 1
].index

print("Unique invoice IDs:", len(unique_invoice_ids))
print(
    "Ambiguous invoice IDs:",
    (invoice_id_counts > 1).sum()
)

Unique invoice IDs: 17640
Ambiguous invoice IDs: 196


In [27]:
safe_invoices = invoices[
    invoices["invoice_id"].isin(unique_invoice_ids)
].copy()

safe_payments = payments[
    payments["invoice_id"].isin(unique_invoice_ids)
].copy()

In [28]:
safe_payment_summary = (
    safe_payments
    .groupby("invoice_id")
    .agg(
        total_paid=("payment_amount", "sum"),
        payment_count=("payment_id", "count")
    )
    .reset_index()
)

safe_invoice_payment_validation = safe_invoices.merge(
    safe_payment_summary,
    on="invoice_id",
    how="left"
)

safe_invoice_payment_validation["total_paid"] = (
    safe_invoice_payment_validation["total_paid"]
    .fillna(0)
)

safe_invoice_payment_validation["payment_gap"] = (
    safe_invoice_payment_validation["grand_total"]
    - safe_invoice_payment_validation["total_paid"]
)

In [29]:
safe_status_summary = (
    safe_invoice_payment_validation
    .groupby("payment_status")
    .agg(
        invoice_count=("invoice_id", "count"),
        avg_invoice_value=("grand_total", "mean"),
        avg_total_paid=("total_paid", "mean"),
        avg_payment_gap=("payment_gap", "mean")
    )
    .round(2)
)

safe_status_summary

,invoice_count,avg_invoice_value,avg_total_paid,avg_payment_gap
payment_status,,,,
Paid,12264,1618975.02,1618975.02,0.00
Partially Paid,3578,1628307.81,821750.79,806557.03
Unpaid,1798,1623160.69,0.00,1623160.69


### Payment-to-Invoice Relationship Validation

Initial payment aggregation produced unexpected results, including apparent overpayments on `Paid` invoices and non-zero payments on some `Unpaid` invoices.

Further investigation showed that these inconsistencies were caused by duplicated `invoice_id` values in the invoice table. Because the payments table links to invoices using only `invoice_id`, payments associated with an ambiguous invoice ID could be incorrectly matched to more than one invoice record.

To validate the underlying payment logic, invoice IDs that uniquely identified one invoice were analyzed separately.

For uniquely identifiable invoices:

- `Paid` invoices had an average payment gap of 0.00.
- `Partially Paid` invoices had a positive remaining payment gap.
- `Unpaid` invoices had an average amount paid of 0.00.

This confirms that payment behavior is internally consistent when the invoice relationship is unambiguous.

Ambiguous invoice IDs will therefore be retained in the dataset but excluded from analyses that require reliable invoice-to-payment linkage.

In [30]:
# Identify invoice IDs that occur more than once
ambiguous_invoice_ids = (
    invoices["invoice_id"]
    .value_counts()
    .loc[lambda x: x > 1]
    .index
)

# Add an ambiguity flag to invoices
invoices["invoice_id_ambiguous"] = (
    invoices["invoice_id"].isin(ambiguous_invoice_ids)
)

# Add the same relationship warning to payments
payments["invoice_id_ambiguous"] = (
    payments["invoice_id"].isin(ambiguous_invoice_ids)
)

In [31]:
ambiguous_payment_ids = (
    payments["payment_id"]
    .value_counts()
    .loc[lambda x: x > 1]
    .index
)

payments["payment_id_ambiguous"] = (
    payments["payment_id"].isin(ambiguous_payment_ids)
)

In [32]:
print(
    "Invoice rows with ambiguous invoice IDs:",
    invoices["invoice_id_ambiguous"].sum()
)

print(
    "Payment rows linked to ambiguous invoice IDs:",
    payments["invoice_id_ambiguous"].sum()
)

print(
    "Payment rows with ambiguous payment IDs:",
    payments["payment_id_ambiguous"].sum()
)

Invoice rows with ambiguous invoice IDs: 393
Payment rows linked to ambiguous invoice IDs: 417
Payment rows with ambiguous payment IDs: 403


### Identifier Quality Flags

Quality flags were created to preserve transaction records while identifying potentially unsafe relationships.

The results show:

- 393 invoice rows contain an ambiguous `invoice_id`.
- 417 payment rows are linked to ambiguous invoice IDs.
- 403 payment rows contain an ambiguous `payment_id`.

These records will remain available for analyses where the transaction itself is still valid.

However, records with ambiguous invoice IDs will be excluded from analyses requiring a reliable one-to-one invoice-to-payment relationship.

This flag-based approach prevents unnecessary data loss while making downstream analysis safer and more transparent.

In [33]:
branches_clean = branches.copy()
customers_clean = customers.copy()
inventory_clean = inventory.copy()
invoices_clean = invoices.copy()
payments_clean = payments.copy()
products_clean = products.copy()
po_header_clean = po_header.copy()
po_lines_clean = po_lines.copy()
so_header_clean = so_header.copy()
so_lines_clean = so_lines.copy()
stock_ledger_clean = stock_ledger.copy()
suppliers_clean = suppliers.copy()

In [34]:
clean_datasets = {
    "branches": branches_clean,
    "customers": customers_clean,
    "inventory": inventory_clean,
    "invoices": invoices_clean,
    "payments": payments_clean,
    "products": products_clean,
    "po_header": po_header_clean,
    "po_lines": po_lines_clean,
    "so_header": so_header_clean,
    "so_lines": so_lines_clean,
    "stock_ledger": stock_ledger_clean,
    "suppliers": suppliers_clean
}

In [35]:
for name in datasets:
    original_rows = len(datasets[name])
    cleaned_rows = len(clean_datasets[name])

    print(
        f"{name}: original={original_rows}, "
        f"cleaned={cleaned_rows}, "
        f"difference={cleaned_rows - original_rows}"
    )

branches: original=6, cleaned=6, difference=0
customers: original=500, cleaned=500, difference=0
inventory: original=180, cleaned=180, difference=0
invoices: original=18033, cleaned=18033, difference=0
payments: original=19257, cleaned=19257, difference=0
products: original=30, cleaned=30, difference=0
po_header: original=24000, cleaned=24000, difference=0
po_lines: original=155495, cleaned=155495, difference=0
so_header: original=20000, cleaned=20000, difference=0
so_lines: original=130402, cleaned=130402, difference=0
stock_ledger: original=237230, cleaned=237230, difference=0
suppliers: original=8, cleaned=8, difference=0


In [36]:
print("FINAL CLEANING VALIDATION")
print("-" * 60)

for name, df in clean_datasets.items():
    print(
        f"{name}: "
        f"rows={len(df)}, "
        f"columns={df.shape[1]}, "
        f"exact_duplicates={df.duplicated().sum()}"
    )

FINAL CLEANING VALIDATION
------------------------------------------------------------
branches: rows=6, columns=14, exact_duplicates=0
customers: rows=500, columns=15, exact_duplicates=0
inventory: rows=180, columns=8, exact_duplicates=0
invoices: rows=18033, columns=12, exact_duplicates=0
payments: rows=19257, columns=8, exact_duplicates=0
products: rows=30, columns=23, exact_duplicates=0
po_header: rows=24000, columns=10, exact_duplicates=0
po_lines: rows=155495, columns=9, exact_duplicates=0
so_header: rows=20000, columns=11, exact_duplicates=0
so_lines: rows=130402, columns=9, exact_duplicates=0
stock_ledger: rows=237230, columns=9, exact_duplicates=0
suppliers: rows=8, columns=12, exact_duplicates=0


In [37]:
print("\nImportant quality flags")
print("-" * 60)

print(
    "Ambiguous invoice rows:",
    invoices_clean["invoice_id_ambiguous"].sum()
)

print(
    "Payments linked to ambiguous invoice IDs:",
    payments_clean["invoice_id_ambiguous"].sum()
)

print(
    "Ambiguous payment ID rows:",
    payments_clean["payment_id_ambiguous"].sum()
)


Important quality flags
------------------------------------------------------------
Ambiguous invoice rows: 393
Payments linked to ambiguous invoice IDs: 417
Ambiguous payment ID rows: 403


# Week 2 — Cleaning Summary

## Cleaning Decisions

The cleaning process focused on preserving valid transactions while improving data reliability and analytical usability.

Key cleaning actions included:

- Converted date fields into proper datetime format.
- Created a numeric `warehouse_capacity_sqft` field while preserving the original warehouse-capacity text field.
- Preserved cancelled purchase orders with missing `received_date` values because the missingness is logically valid.
- Preserved duplicate invoice and payment transactions because they are not exact duplicate rows.
- Created unique analytical record identifiers for invoices and payments.
- Added ambiguity flags for duplicated invoice and payment identifiers.
- Validated product, inventory, purchase-order, sales-order, invoice, payment, and financial business rules.
- Reconciled purchase-order and sales-order header totals with their underlying line items.
- Validated invoice totals against linked sales orders.
- Confirmed that payment logic behaves correctly when invoice identifiers are unambiguous.
- Preserved all original rows rather than deleting valid business records.

## Important Data Quality Limitation

Duplicated `invoice_id` values create ambiguity when linking invoices to payments because the payments table only contains `invoice_id`.

Records with ambiguous invoice identifiers should therefore be excluded from analyses requiring reliable invoice-to-payment reconciliation.

They may still be retained for other valid invoice, customer, and sales analyses.

## Week 2 Conclusion

The datasets are now structurally validated and suitable for creating analysis-ready datasets.

No valid transaction records were removed during cleaning.

The next project phase will focus on data integration, feature engineering, and preparation of analytical datasets for product, inventory, warehouse, supplier, customer, and predictive analysis.

In [38]:
for name, df in clean_datasets.items():
    output_file = f"processed_data/{name}_clean.csv"
    df.to_csv(output_file, index=False)

print("Cleaned datasets exported successfully.")

Cleaned datasets exported successfully.


In [39]:
import os

os.listdir("processed_data")

['branches_clean.csv',
 'customers_clean.csv',
 'inventory_clean.csv',
 'invoices_clean.csv',
 'payments_clean.csv',
 'po_header_clean.csv',
 'po_lines_clean.csv',
 'products_clean.csv',
 'so_header_clean.csv',
 'so_lines_clean.csv',
 'stock_ledger_clean.csv',
 'suppliers_clean.csv']